# Figure 1a — single-molecule PSFs on a patterned (Bayer) and a non-patterned camera

One frame of a single ATTO 488, ATTO 565 and ATTO 647N molecule, simulated on

* a **patterned** camera: the Ximea's RGGB Bayer mosaic with its measured B/G/R pixel QE curves, and
* a **non-patterned** camera: the same pixels with no colour filters, every pixel with one broad QE
  curve (peak = the Bayer green peak; flat over the dyes' emission), as in the original panel.

On the patterned camera the dye's colour shows up as a pixel-to-pixel pattern within the PSF; on the
non-patterned camera the three dyes look alike.

Each image is simulated photon by photon with the vectorial dipole PSF (pyS3M's default,
`psf_model="dipole"`): every photon gets a wavelength from the dye's emission spectrum, lands where
the dipole PSF at that wavelength puts it, and is detected with its pixel's QE at that wavelength.
Background photons, read noise and gain are those of the Ximea calibration (median per-pixel values).

Images are shown in photoelectrons, ((ADU − offset) / gain). All six panels share one colour
scale (bar on the ATTO 647N panels). The 2×2 key in the top-left corner shows the filters of the pixels it covers, taken from the
masks each image was simulated with (B/G/R for the Bayer camera, grey = unfiltered).

Runs in seconds; no data download. SVGs are written to `figures/` (git-ignored).


In [ ]:
import logging
import types
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pyS3M import IOFunctions, MaskFunctions, PlottingBase, SpectralFunctions, sCMOSFunctions
from pyS3M.simulation.multicolour import MultiC_Sim_Funcs

logging.basicConfig(level=logging.INFO, format="%(name)s — %(levelname)s — %(message)s")

IO = IOFunctions.IO_Functions()
M_F = MaskFunctions.Mask_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF = MultiC_Sim_Funcs()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5}, extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack, data_arg="image",
)

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/figures -> repo root
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
FIGURE_DIR = Path("figures")  # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

DYES = ["ATTO 488", "ATTO 565", "ATTO 647N"]
SEEDS = {"ATTO 488": 1, "ATTO 565": 2, "ATTO 647N": 3}  # one fixed noise realisation per dye

IMAGE_SIZE = 10       # px
PIXEL_SIZE = 69       # nm, Ximea
NA = 1.49
N_PHOTONS = 4000      # photons emitted into the objective per frame
N_BACKGROUND = 4      # mean background photons per pixel per frame
X0_PX, Y0_PX = 4.75, 4.75  # emitter position (column, row) in px, off the pixel centre as in the original panel


## The two cameras

Both use the median gain, offset, variance, read noise and relative QE of the Ximea calibration,
uniform over the 10×10 px image. They differ only in the pixels' QE curves and the mosaic.


In [ ]:
calibration = {k: float(np.median(IO.read_tiff(str(CALIBRATION_DIR / f"{k}.tif"))))
               for k in ["gain", "offset", "variance", "readnoise", "rqe"]}
print({k: round(v, 3) for k, v in calibration.items()})

R, G, B, wavelength = S_F.getpixelefficiency()  # Ximea B/G/R pixel QE curves


def broad_qe(wl, peak, centre=650.0, width=500.0):
    # one smooth, nearly flat QE curve for an unfiltered pixel
    return peak * np.exp(-0.5 * (wl - centre) ** 2 / width ** 2)


def make_camera(masks, pixel_QYs, pixel_order):
    cam = {k: np.full((IMAGE_SIZE, IMAGE_SIZE), calibration[k]) for k in ["gain", "offset", "variance", "rqe"]}
    cam["readnoise"] = calibration["readnoise"]
    cam.update(masks=masks, pixel_QYs=pixel_QYs, pixel_order=pixel_order,
               pixel_order_indices={c: i for i, c in enumerate(pixel_order)})
    return cam


CAMERAS = {
    "Bayer": make_camera(M_F.get_masks(size_x=IMAGE_SIZE, size_y=IMAGE_SIZE),  # RGGB
                         np.vstack([B, G, R]), ["B", "G", "R"]),
    "NonPatterned": make_camera(M_F.get_masks(size_x=IMAGE_SIZE, size_y=IMAGE_SIZE,
                                              mosaic_unit=np.array([["W", "W"], ["W", "W"]])),
                                broad_qe(wavelength, np.max(G))[None, :], ["W"]),
}
KEY_COLOURS = {"Bayer": list(PlottingBase.BAYER_COLOURS), "NonPatterned": ["#c0c0c0"]}

fig, ax = plt.subplots(figsize=(1.8, 1.2))
for qe, colour in zip([B, G, R], PlottingBase.BAYER_COLOURS):
    ax.plot(wavelength, qe, color=colour)
ax.plot(wavelength, CAMERAS["NonPatterned"]["pixel_QYs"][0], color="k", ls="--")
ax.set_xlim(400, 900)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("QE")
plt.show()


## Simulate one frame per dye and camera

In [ ]:
def simulate(dye, cam, seed):
    # one frame of one molecule, in photoelectrons
    aew, dpe = S_F.get_pixel_fractions_dye_and_filters(dye, [], wavelength, cam["pixel_QYs"])
    x0y0 = {dye: np.array([[X0_PX * PIXEL_SIZE], [Y0_PX * PIXEL_SIZE]])[None]}  # (1, 2, 1), (x, y) in nm
    np.random.seed(seed)
    img, _, _ = MSF.gen_camera_image_stack(
        cam, wavelength, np.array(aew), np.array(dpe),
        {dye: np.full(1, N_PHOTONS)}, x0y0,
        background_photons=N_BACKGROUND,
        background_colour=[1.0] * len(cam["pixel_order"]),
        smoothing_function=smoothing_function,
        NA=NA, pixel_size=PIXEL_SIZE,
        emission_spectra=MSF.emission_spectra_for(x0y0, [], wavelength),
    )
    return (np.squeeze(img) - cam["offset"]) / cam["gain"]  # ADU -> photoelectrons


images = {(dye, name): simulate(dye, cam, SEEDS[dye]) for dye in DYES for name, cam in CAMERAS.items()}
for (dye, name), img in images.items():
    print(f"{dye:10s} {name:13s} total {img.sum():6.0f} pe, peak {img.max():5.0f} pe")


## Panels

One SVG per panel, `<dye>_<camera>.svg`. All six share one photoelectron scale, so the dimmer Bayer
panels (fewer photoelectrons detected through the colour filters) look dimmer. The scale's colour bar is
drawn on the panels of `COLOURBAR_DYE` only; the other panels keep its space empty, so all six images
are the same size when the figure is assembled.

In [ ]:
COLOURBAR_DYE = "ATTO 647N"  # the panels (rightmost column of Fig. 1a) that carry the shared colour bar
VMIN = min(img.min() for img in images.values())
VMAX = max(img.max() for img in images.values())
print(f"shared scale: {VMIN:.0f} to {VMAX:.0f} photoelectrons")


def draw(ax, dye, name):
    plotter.image_plot(ax, images[(dye, name)], vmin=VMIN, vmax=VMAX,
                       colorbar=True, colorbar_label="photoelectrons",
                       scalebar=True, pixelsize=PIXEL_SIZE, scalebarsize=200, scalebarlabel="200 nm",
                       scalebar_color="white")
    PlottingBase.add_bayer_key(ax, CAMERAS[name]["masks"], pixel_order=CAMERAS[name]["pixel_order"],
                               colours=KEY_COLOURS[name])


for dye in DYES:
    for name in CAMERAS:
        fig, ax = plt.subplots(figsize=(1.3, 1.0))
        draw(ax, dye, name)
        fig.canvas.draw()                # lay out with the colour bar in place ...
        fig.set_layout_engine("none")    # ... and keep that layout, so every image has the same size
        if dye != COLOURBAR_DYE:
            for cax in ax.child_axes:    # the colour bar (an inset of the image axes)
                cax.set_visible(False)
        fig.savefig(FIGURE_DIR / f"{dye.replace(' ', '_')}_{name}.svg", dpi=600, format="svg")
        plt.close(fig)

# overview of the six panels (not saved): rows = camera, columns = dye
fig, axs = plt.subplots(2, 3, figsize=(4.2, 2.2))
for col, dye in enumerate(DYES):
    for row, name in enumerate(CAMERAS):
        draw(axs[row, col], dye, name)
fig.canvas.draw()
fig.set_layout_engine("none")
for ax in axs[:, :2].ravel():
    for cax in ax.child_axes:
        cax.set_visible(False)
plt.show()
print("saved:", sorted(p.name for p in FIGURE_DIR.glob("ATTO_*_*.svg")))
